# Convex-hull and phase-diagram examples

This notebook exercises the public generic and materials-science APIs. It uses Matplotlib's non-interactive Agg backend and closes its figure, so it performs no display or file-writing side effects.

In [ ]:
from httk.analyse.generic import LowerConvexHull

hull = LowerConvexHull(
    [(0.0,), (0.5,), (1.0,)],
    [0.0, -1.0, 0.0],
)

hull_indices = tuple(hull.hull_indices)
above_hull = hull.value_above_hull
middle_decomposition = hull.decomposition(1)
segments = hull.supported_segments

assert hull_indices == (0, 1, 2)
assert above_hull[1] == 0.0

## A binary phase diagram

`PhaseDiagram` normalizes these formula-unit compositions and compares their per-atom energies.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from httk.analyse.matsci import PhaseDiagram

diagram = PhaseDiagram.from_compositions(
    [{"A": 1}, {"B": 1}, {"A": 1, "B": 1}],
    [0.0, 0.0, -2.0],
    ids=["A", "B", "AB"],
)

phase_indices = tuple(diagram.hull_indices)
above_hull = diagram.energy_above_hull
phase_lines = diagram.phase_lines

assert phase_indices == (0, 1, 2)
assert above_hull[2] == 0.0

ax = diagram.plot()
plt.close(ax.figure)

## Incremental building and unknown energy

`PhaseDiagramBuilder` supports chained additions. A phase with `energy=None` is retained for plotting but does not affect the known-energy hull.

In [ ]:
from httk.analyse.matsci import PhaseDiagramBuilder

builder = (
    PhaseDiagramBuilder()
    .add_phase({"A": 1}, 0.0, "A")
    .add_phase({"B": 1}, 0.0, "B")
    .add_phase({"A": 1, "B": 1}, -2.0, "AB")
    .add_phase({"A": 1, "B": 2}, None, "unknown-AB2")
)
built = builder.build()

assert built.unknown_ids == ("unknown-AB2",)
ax = built.plot()
assert any(line.get_marker() == "s" for line in ax.lines)
plt.close(ax.figure)